In [ ]:
import matplotlib.pyplot as plt
from adaptive_latents import ArrayWithTime
from adaptive_latents.input_sources import LDS
from adaptive_latents import VJF, Bubblewrap, StreamingKalmanFilter
from adaptive_latents.regressions import BaseKNearestNeighborRegressor
from adaptive_latents.stim_regressor import StimRegressor
import numpy as np
from tqdm.notebook import trange
import functools
import copy

rng = np.random.default_rng()


In [ ]:
transitions_per_rotation = 30 + 1/np.pi
stims_per_rotation = 3
n_rotations = 250
initial_state = np.array([100,0,0], dtype=np.float64)
assert stims_per_rotation < transitions_per_rotation/2


lds = LDS.nest_lds(transitions_per_rotation=transitions_per_rotation, rng=rng)

In [ ]:
def S(state, stim):
    # return stim* np.array([0,0, np.exp(-state @ state/60**2)]) * 3
    # return stim* np.array([0,0, np.cos(state[0]/90)]) * 3
    return stim*np.array([0, state[0]/np.linalg.norm(state[:2]), 0]) * 3
    # return np.ones(3)*stim*10 + rng.normal(size=3)
    # return stim*np.array([np.sin(state[0] / 5), np.sin(state[1]/5), np.sin(state[2]/5)]) * 10


def U(lds, state, i, rng):
    stim = float(rng.random() < stims_per_rotation/transitions_per_rotation)
    u = S(state, stim)
    return u

alternate_histories = []
for _ in trange(100):
    X, _, _ = copy.deepcopy(lds).simulate(n_steps=int(n_rotations*transitions_per_rotation), initial_state=initial_state, U=U, rng=rng)
    alternate_histories.append(X)
alternate_histories = np.array(alternate_histories)
originally_sampled_points = rng.choice(alternate_histories.reshape((-1,3)), size=1_000, replace=False)

def evaluate(function):
    ret = []
    for point in originally_sampled_points:
        ret.append(function(np.hstack([point, [1]])))
    return np.array(ret)

In [ ]:


sr = StimRegressor(autoreg=StreamingKalmanFilter(), stim_reg=BaseKNearestNeighborRegressor(k=20, maxlen=1000), attempt_correction=True)

state = initial_state

observations = []
stims = []
predictions = []
dt_X = []

s_hat_evals = []
a_errors = []
full_iteration = None


n_steps = int(n_rotations * transitions_per_rotation)
for i in trange(n_steps):
    state, observation, delivered_stim = lds.simulate_step(state, rng, u_function=U, i=i, use_state_dynamics=i != 0, add_centers=False)

    stim = (delivered_stim).any()

    qX = ArrayWithTime([[1]], i)
    sr.partial_fit_transform(np.array([[stim]]), stream='stim')
    prediction = sr.partial_fit_transform(qX, stream='dt_X')
    sr.partial_fit_transform(observation[None,:], stream='X')


    observations.append(ArrayWithTime(observation, i))
    stims.append(ArrayWithTime(stim, i))
    predictions.append(prediction)
    dt_X.append(qX)

    if stim and (sr.autoreg.get_arbitrary_dynamics_parameter() is not None):
        e = evaluate(sr.stim_reg.predict)
        s_hat_evals.append(ArrayWithTime(e, i))
        a_errors.append(ArrayWithTime((sr.autoreg.A - lds.A)**2, i))
        if not np.isnan(sr.stim_reg.history).any() and full_iteration is None:
            full_iteration = i

observations = ArrayWithTime.from_list(observations, squeeze_type='to_2d')
stims = ArrayWithTime.from_notime(stims)
# predictions = ArrayWithTime.from_list(predictions, drop_early_nans=True)
dt_X = ArrayWithTime.from_list(dt_X, squeeze_type='to_2d')

s_hat_evals = ArrayWithTime.from_list(s_hat_evals, drop_early_nans=True)
a_errors = ArrayWithTime.from_list(a_errors)

s_eval = evaluate(functools.partial(S, stim=1))

In [ ]:
%matplotlib qt
fig, axs = plt.subplots(nrows=2, figsize=(10,5))
ax = axs[0]

s_hat_errors = s_hat_evals - s_eval

mse_per_eval = (s_hat_errors**2).mean(axis=1)

ax.plot(s_hat_evals.t, mse_per_eval, '.-')


if (s_eval != 0).any():
    zero_estimator_mse = ((0-s_eval)**2).mean(axis=0)
    for idx, line in enumerate(zero_estimator_mse):
        ax.axhline(line, color=f'C{idx}', alpha=.3)

if full_iteration is not None:
    ax.axvline(full_iteration, color='k', alpha=.3)

ax = axs[1]
ax.plot(a_errors.t, np.log(np.linalg.norm(a_errors, axis=(1,2))), 'C3--', alpha=.5);




In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(12,6), sharex=True, sharey=True)

scatters = []

c = [S(p, 1)[2] for p in originally_sampled_points]
s = axs[0].scatter(originally_sampled_points[:,0], originally_sampled_points[:,1], c=c)
scatters.append(s)

s = axs[1].scatter(sr.stim_reg.history[:,0], sr.stim_reg.history[:,1], c=sr.stim_reg.history[:,5] )
scatters.append(s)


clim = np.array([s.get_clim() for s in scatters])
clim = (clim[:,0].min(), clim[:,1].max())
for s in scatters:
    s.set_clim(clim)


fig.colorbar(s, ax = axs);


In [ ]:
plt.scatter(observations[:,0], observations[:,1], c=observations.t, cmap='viridis')

In [ ]:
fig, ax = plt.subplots()

ax.plot(np.arange(alternate_histories.shape[1])/transitions_per_rotation, np.linalg.norm(alternate_histories, axis=-1).T)
ax.set_xlabel('rotations')